# 📈 Stock Price Prediction using Machine Learning
**Task 2 - ML Internship**

In this notebook, we will:
- Fetch real stock data from Yahoo Finance
- Prepare and explore the data
- Train a Linear Regression AND a Random Forest model
- Compare their performance
- Plot Actual vs Predicted closing prices

## Step 1: Install and Import Libraries
First, we install `yfinance` (if not already installed) and import everything we need.

In [ ]:
# Install yfinance if you haven't already
# Run this cell once, then you can comment it out
!pip install yfinance scikit-learn matplotlib pandas

In [ ]:
# Importing all the libraries we need
import yfinance as yf                        # To fetch stock data from Yahoo Finance
import pandas as pd                          # For data manipulation
import numpy as np                           # For numerical operations
import matplotlib.pyplot as plt              # For plotting graphs

from sklearn.linear_model import LinearRegression       # Linear Regression model
from sklearn.ensemble import RandomForestRegressor      # Random Forest model
from sklearn.model_selection import train_test_split    # To split data into train/test
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Evaluation metrics

import warnings
warnings.filterwarnings('ignore')  # Just to keep the output clean

print("All libraries imported successfully!")

## Step 2: Fetch Stock Data
We will use Apple (AAPL) as our stock. You can change the ticker to any stock you like, for example:
- `TSLA` → Tesla
- `GOOGL` → Google
- `MSFT` → Microsoft
- `AMZN` → Amazon

In [ ]:
# ---- CHANGE THIS if you want a different stock ----
STOCK_TICKER = "AAPL"   # Apple Inc.
START_DATE   = "2020-01-01"
END_DATE     = "2024-12-31"
# ---------------------------------------------------

# Downloading the stock data
print(f"Fetching data for {STOCK_TICKER}...")
df = yf.download(STOCK_TICKER, start=START_DATE, end=END_DATE)

print(f"\nData downloaded! Shape: {df.shape}")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
df.head()

## Step 3: Explore the Data
Let's take a look at what we are working with before jumping into the model.

In [ ]:
# Basic info about the dataset
print("=== Dataset Info ===")
print(df.info())

print("\n=== Basic Statistics ===")
print(df.describe().round(2))

In [ ]:
# Check for any missing values
print("Missing values in each column:")
print(df.isnull().sum())

# Drop missing values if any
df.dropna(inplace=True)
print("\nAll good! No missing values remaining.")

In [ ]:
# Plot the closing price over time to get a feel for the data
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], color='steelblue', linewidth=1.5)
plt.title(f'{STOCK_TICKER} Closing Price Over Time', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 4: Prepare Features and Target

We want to **predict the next day's closing price** using today's data.

- **Features (X):** Open, High, Low, Volume (today's values)
- **Target (y):** Close price of the **next day**

We do this by shifting the Close column by -1 (moving it one step up).

In [ ]:
# Flatten MultiIndex columns if yfinance returned them
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Create the target column: next day's closing price
# shift(-1) moves everything up by 1 row, so today's features map to tomorrow's close
df['Next_Close'] = df['Close'].shift(-1)

# Drop the last row because it has no next day value (NaN)
df.dropna(inplace=True)

# Define our features and target
FEATURES = ['Open', 'High', 'Low', 'Volume']
TARGET   = 'Next_Close'

X = df[FEATURES]
y = df[TARGET]

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print("\nFirst few rows of features:")
X.head()

## Step 5: Split into Train and Test Sets

We use 80% of the data for training and 20% for testing.

**Important:** We use `shuffle=False` because this is time series data — the order matters!

In [ ]:
# Split the data — no shuffling for time series!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")

## Step 6: Train the Models

We'll train **two models** and compare them:
1. **Linear Regression** — simple, fast, good baseline
2. **Random Forest** — more powerful, handles non-linear patterns better

In [ ]:
# ---- Model 1: Linear Regression ----
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)
print("Linear Regression model trained!")

# ---- Model 2: Random Forest ----
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
print("Random Forest model trained!")

## Step 7: Evaluate the Models

We use 3 common metrics:
- **MAE** (Mean Absolute Error) — average error in dollars
- **RMSE** (Root Mean Squared Error) — penalizes big errors more
- **R² Score** — how well the model explains the data (1.0 = perfect)

In [ ]:
def evaluate_model(name, y_true, y_pred):
    """Prints evaluation metrics for a given model."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"--- {name} ---")
    print(f"  MAE  : ${mae:.2f}")
    print(f"  RMSE : ${rmse:.2f}")
    print(f"  R²   : {r2:.4f}")
    print()

evaluate_model("Linear Regression", y_test, lr_predictions)
evaluate_model("Random Forest",     y_test, rf_predictions)

## Step 8: Plot Actual vs Predicted Prices

Now the fun part — let's visualize how well our models performed!

In [ ]:
# Get the dates for the test set
test_dates = df.index[len(X_train):]

# ---- Plot: Linear Regression ----
plt.figure(figsize=(14, 5))
plt.plot(test_dates, y_test.values,    label='Actual Close',             color='black',      linewidth=1.5)
plt.plot(test_dates, lr_predictions,   label='Linear Regression (Pred)', color='dodgerblue', linewidth=1.2, linestyle='--')
plt.title(f'{STOCK_TICKER} - Actual vs Predicted (Linear Regression)', fontsize=15)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot: Random Forest ----
plt.figure(figsize=(14, 5))
plt.plot(test_dates, y_test.values,  label='Actual Close',           color='black',       linewidth=1.5)
plt.plot(test_dates, rf_predictions, label='Random Forest (Pred)',   color='tomato',      linewidth=1.2, linestyle='--')
plt.title(f'{STOCK_TICKER} - Actual vs Predicted (Random Forest)', fontsize=15)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot: Both Models Together ----
plt.figure(figsize=(14, 6))
plt.plot(test_dates, y_test.values,  label='Actual Close',           color='black',      linewidth=2)
plt.plot(test_dates, lr_predictions, label='Linear Regression',      color='dodgerblue', linewidth=1.2, linestyle='--')
plt.plot(test_dates, rf_predictions, label='Random Forest',          color='tomato',     linewidth=1.2, linestyle='--')
plt.title(f'{STOCK_TICKER} - Actual vs Predicted (Both Models)', fontsize=15)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 9: Feature Importance (Random Forest)

Random Forest can tell us which features were most useful for making predictions.

In [ ]:
# Get feature importances from Random Forest
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({'Feature': FEATURES, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values('Importance', ascending=True)

# Plot
plt.figure(figsize=(8, 4))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='steelblue')
plt.title('Feature Importance - Random Forest', fontsize=14)
plt.xlabel('Importance Score')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print(feat_imp_df.sort_values('Importance', ascending=False))

## Step 10: Predict Tomorrow's Closing Price

Let's use the most recent data point to predict what the next closing price might be.

In [ ]:
# Take the last available row as our "today" input
latest_data = X.iloc[[-1]]  # using double brackets to keep it as a DataFrame

print("Latest available data (used as input):")
print(latest_data.to_string())

# Predict using both models
lr_next = lr_model.predict(latest_data)[0]
rf_next = rf_model.predict(latest_data)[0]

print(f"\n📌 Predicted Next Closing Price:")
print(f"   Linear Regression : ${lr_next:.2f}")
print(f"   Random Forest     : ${rf_next:.2f}")

## Summary

| Model | What it does | Best for |
|---|---|---|
| **Linear Regression** | Finds a straight-line relationship between features and price | Simple baseline, fast |
| **Random Forest** | Builds many decision trees and averages their results | Handles complex patterns better |

### Key Takeaways:
- Random Forest generally performs better for stock prediction because markets are non-linear
- The features `High` and `Low` tend to be the most important for predicting next day's close
- Stock prediction is inherently uncertain — no model will be 100% accurate!
- This model can be improved by adding more features like moving averages, RSI, etc.

---
*Task 2 Complete ✅*